# Regularisation and Normalisation in Torch

#### Overview
In this segment, we explore how different regularisation and normalisation techniques help improve the performance and generalisation of neural networks. You’ll learn how dropout, batch normalisation, L2 regularisation, and early stopping reduce overfitting and stabilise training in PyTorch.

#### Agenda
- **Dropout (`nn.Dropout`)** → randomly switches off neurons during training to prevent overfitting.  
- **Batch Normalisation (`nn.BatchNorm1d`)** → normalises activations to speed up and stabilise training.  
- **L2 Regularisation (`weight_decay` in `optim.SGD`)** → penalises large weights to keep them small and stable.  
- **Early Stopping (validation-based)** → stops training when validation loss stops improving.  
- **Application to MNIST** → helps the MLP generalise better and achieve higher accuracy.
  
#### Application to MNIST
These techniques help the MLP generalise better, avoid memorising training data, and achieve higher accuracy on the test set.

#### Key Takeaway
By the end of this segment, you’ll understand how combining dropout, batch normalisation, L2 regularisation, and early stopping can make your MNIST MLP model more robust, prevent overfitting, and improve test performance — achieving up to **98% accuracy**.

#### Importing Required Libraries ####

- **`import torch`** → loads the core PyTorch library for creating tensors and running computations on CPU/GPU.  
- **`import torch.nn as nn`** → imports PyTorch’s neural network module, which provides building blocks like layers (`Linear`, `Conv2d`, etc.).  
- **`import torch.nn.functional as F`** → imports a collection of functions (e.g., activation functions like `ReLU`, `softmax`, loss functions) that can be used directly.
- **`import torch.optim as optim`** → loads PyTorch’s optimisation module. Provides algorithms like **SGD** and **Adam** to update model weights during training.  
- **`import pickle`** → used to load and save Python objects; in this context, it helps load the MNIST dataset stored in a `.pkl` file.  

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pickle
from torch.utils.data import Dataset, DataLoader

#### MLP with Dropout and Batch Normalisation
This `MLP` class defines a feedforward neural network with two fully connected layers and additional techniques to improve training:  

- **fc1**: first fully connected layer that maps the input to a hidden layer.  
- **BatchNorm (bn1)**: normalises activations in the hidden layer to stabilise and speed up learning.  
- **Dropout**: randomly drops some hidden units during training to reduce overfitting.  
- **fc2**: second fully connected layer that maps the hidden layer to the final outputs (logits).  

**Forward pass**: the input passes through `fc1`, then batch normalisation and ReLU activation, followed by dropout, and finally through `fc2` to produce the output scores.  

In [13]:
# Define a Multi-Layer Perceptron (MLP) model
class MLP1(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.5):
        super(MLP1, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)   # Normalisation
        self.dropout = nn.Dropout(dropout_rate) # Dropout
        self.fc2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))   # input → hidden
        x = self.dropout(x)                 # dropout
        x = self.fc2(x)                     # hidden → output
        return x


### Loading the MNIST Data

The MNIST dataset is loaded from a pickle file and split into three parts: **training**, **validation**, and **test** sets.  
- Each set contains images (`x`) and labels (`y`).  
- Images (`train_x`, `valid_x`, `test_x`) are converted into `float32` tensors for model input.  
- Labels (`train_y`, `valid_y`, `test_y`) are converted into `long` tensors, which is the required type for classification tasks in PyTorch.  

This gives us separate datasets for training the model, validating performance during training, and testing on unseen data.  

In [29]:
class CustomMNISTDataset(Dataset):

    # defining constructor
    def __init__(self, path, tag="train", flatten=True):

        if tag == "train":
            images, labels = pickle.load(open(path, "rb"), encoding="latin1")[0]  # load train set

        elif tag == "valid":
            images, labels = pickle.load(open(path, "rb"), encoding="latin1")[1]  # load validation set
            
        elif tag == "test":
            images, labels = pickle.load(open(path, "rb"), encoding="latin1")[2]  # load test set
        
        x = torch.tensor(images, dtype=torch.float32)  # convert images to float32 tensor
        y = torch.tensor(labels, dtype=torch.long)     # convert labels to int64 tensor

        if flatten:                                    # flatten 28x28 → 784
            x = x.view(x.shape[0], -1)

        # set the class/dataset attributes
        self.x, self.y = x, y  # store tensors and optional transform

    # defining what is the length
    def __len__(self): 
        return len(self.y)                             # return total number of samples

    # defining how to get idx item
    def __getitem__(self, idx):
        img = self.x[idx]                              # get one image
        
        return img, self.y[idx]                        # return image and label


train_dataset = CustomMNISTDataset("data/mnist.pkl", tag="train", flatten=True)
valid_dataset = CustomMNISTDataset("data/mnist.pkl", tag="valid", flatten=True)
test_dataset = CustomMNISTDataset("data/mnist.pkl", tag="test", flatten=True)


trainloader = DataLoader(train_dataset, batch_size=32)
validloader = DataLoader(valid_dataset, batch_size=32)
testloader = DataLoader(test_dataset, batch_size=32)

#### Model, Loss, and Optimiser Setup
- The model is defined with an input size of 784 (flattened 28×28 images), a hidden layer of 128 neurons, and an output size of 10 classes for digits 0–9.  
- The `MLP` is created with a dropout rate of 0.3.  
- **CrossEntropyLoss** is used as the loss function for multi-class classification.  
- The **SGD optimiser** is configured with a learning rate of 0.1 and **L2 regularisation** (`weight_decay=1e-4`) to penalise large weights and reduce overfitting.  


In [18]:
# Model, loss, optimizer
input_size = 784
hidden_size = 128
output_size = 10

model_reg = MLP1(input_size, hidden_size, output_size, dropout_rate=0.3)
criterion = nn.CrossEntropyLoss()

# Using L2 regularisation via weight_decay
optimizer = optim.SGD(model_reg.parameters(), lr=0.1, weight_decay=1e-4)

### Training Loop with Early Stopping

- Training for 30 epochs with mini-batches of size 64.  
- Each batch: forward pass, compute loss, backpropagate, and update weights.  
- After each epoch: evaluating on the validation set, calculate loss and accuracy, and print results.  
- Early stopping: if validation loss improves, save the model; if not improved for 3 epochs, stop and restore the best model.  


In [25]:
# Training loop with Early Stopping
epochs = 10
patience = 3

best_val_loss, no_improve_epochs = float('inf'), 0

for epoch in range(epochs):
    model_reg.train()
    for input_, expected_output_ in trainloader:
        
        loss = criterion(model_reg(input_), expected_output_)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    
    model_reg.eval()
    with torch.no_grad():

        model_input_ = valid_dataset[:][0]
        expected_output_ = valid_dataset[:][1]
        
        model_output_ = model_reg(model_input_)
        
        val_loss = criterion(model_output_, expected_output_)
        
        model_class_ = torch.argmax(F.softmax(model_output_, dim=1), dim=1)
        
        val_acc = (model_class_ == expected_output_).float().mean().item()

    
    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Train Loss: {loss.item():.4f}",
        f"Val Loss: {val_loss.item():.4f}", 
        f"Val Acc: {val_acc:.4f}")

    
    if val_loss.item() < best_val_loss:  # Early stopping
        best_val_loss, no_improve_epochs = val_loss.item(), 0
        best_model_state = model_reg.state_dict()
        
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print("Early stopping triggered.")
            break

print("Saving the best model: ")
model_reg.load_state_dict(best_model_state)

Epoch 1/10 Train Loss: 0.1140 Val Loss: 0.0799 Val Acc: 0.9774
Epoch 2/10 Train Loss: 0.0261 Val Loss: 0.0800 Val Acc: 0.9781
Epoch 3/10 Train Loss: 0.0586 Val Loss: 0.0789 Val Acc: 0.9783
Epoch 4/10 Train Loss: 0.0122 Val Loss: 0.0801 Val Acc: 0.9799
Epoch 5/10 Train Loss: 0.0467 Val Loss: 0.0788 Val Acc: 0.9781
Epoch 6/10 Train Loss: 0.0390 Val Loss: 0.0808 Val Acc: 0.9783
Epoch 7/10 Train Loss: 0.0081 Val Loss: 0.0796 Val Acc: 0.9787
Epoch 8/10 Train Loss: 0.0098 Val Loss: 0.0795 Val Acc: 0.9780
Early stopping triggered.
Saving the best model: 


<All keys matched successfully>

### Evaluation on the Test Set

- Switching the model to evaluation mode with `model.eval()` and disable gradient tracking using `torch.no_grad()`.  
- Running the test data through the model to get output scores.  
- Applying **softmax** to convert scores into probabilities and use **argmax** to pick the predicted class for each sample.  
- Comparing predictions with true labels (`y_test`) to count correct matches and compute overall accuracy.  
- Printing the final test accuracy as a percentage, and also showing a few sample predictions (first 10) with their true labels to see how the model performs on individual cases. 


In [27]:
# Evaluation on test set
model_reg.eval()
with torch.no_grad():

    model_input_ = test_dataset[:][0]
    expected_output_ = test_dataset[:][1]
    
    model_output_ = model_reg(model_input_)
    model_class_ = torch.argmax(F.softmax(model_output_, dim=1), dim=1)

    correct = (model_class_ == expected_output_).sum().item()
    accuracy = correct / len(y_test)

print(f"\nTest Accuracy: {accuracy*100:.2f}%")

print("\nSample Predictions (first 10):")
for i in range(10):
    print(f"True: {expected_output_[i].item()}  Predicted: {model_class_[i].item()}")


Test Accuracy: 97.86%

Sample Predictions (first 10):
True: 7  Predicted: 7
True: 2  Predicted: 2
True: 1  Predicted: 1
True: 0  Predicted: 0
True: 4  Predicted: 4
True: 1  Predicted: 1
True: 4  Predicted: 4
True: 9  Predicted: 9
True: 5  Predicted: 6
True: 9  Predicted: 9


#### Conclusion
  - Builds on MLP by adding Batch Normalisation, Dropout, and L2 regularisation.  
  - Includes early stopping based on validation loss for better generalisation.  
  - These techniques prevent overfitting and stabilise training.  
  - Provides the most balanced and robust results among the three with 98% accuracy.